In [1]:
import dask.dataframe as dd
from dask.distributed import Client, LocalCluster
import joblib

In [2]:
# Set up Dask cluster
cluster = LocalCluster(n_workers=24, processes=True, threads_per_worker=4, memory_limit='200GB')
client = Client(cluster)

/usr/local/lib/python3.10/dist-packages/distributed/node.py:182: UserWarning: Port 8787 is already in use.
Perhaps you already have a cluster running?
Hosting the HTTP server on port 42785 instead
  warnings.warn(


In [8]:
df = dd.read_parquet('/workspace/nemo_datasets/phu_classifier_filtering')

In [9]:
df

,filename,text,id,domain,sentiment,quality_score
npartitions=256,,,,,,
,string,string,string,string,int64,float64
,...,...,...,...,...,...
...,...,...,...,...,...,...
,...,...,...,...,...,...
,...,...,...,...,...,...


In [10]:
df1 = df.sample(frac=50000/len(df), random_state=42).compute()|

In [13]:
df['domain'].value_counts().compute()

domain
Arts_and_Entertainment       941159
Health                       886657
People_and_Society           843624
Sports                       793235
News                         769510
Sensitive_Subjects           743582
Food_and_Drink               675447
Business_and_Industrial      624991
Travel_and_Transportation    581803
Jobs_and_Education           564654
Beauty_and_Fitness           555714
Home_and_Garden              529616
Computers_and_Electronics    516543
Books_and_Literature         396770
Internet_and_Telecom         379943
Law_and_Government           358097
Games                        345954
Autos_and_Vehicles           328610
Real_Estate                  319040
Finance                      304233
Pets_and_Animals             189001
Shopping                     187516
Science                      182682
Hobbies_and_Leisure           98387
Adult                         90323
Online_Communities            52363
Name: count, dtype: int64[pyarrow]

In [11]:
df1['domain'].value_counts()

domain
Arts_and_Entertainment       3904
Health                       3592
People_and_Society           3342
Sports                       3331
News                         3150
Sensitive_Subjects           3064
Food_and_Drink               2791
Business_and_Industrial      2543
Travel_and_Transportation    2333
Beauty_and_Fitness           2277
Home_and_Garden              2216
Jobs_and_Education           2214
Computers_and_Electronics    2119
Books_and_Literature         1652
Internet_and_Telecom         1490
Law_and_Government           1458
Games                        1441
Real_Estate                  1339
Autos_and_Vehicles           1324
Finance                      1243
Pets_and_Animals              765
Shopping                      726
Science                       711
Hobbies_and_Leisure           392
Adult                         378
Online_Communities            204
Name: count, dtype: int64[pyarrow]

In [15]:
df1.to_csv('50k_classifier_filtering.csv',index=False)

In [20]:
import pandas as pd

df2 = dd.read_parquet('phu_heuristic_filtering')

In [21]:
df2

,filename,text,id,domain,sentiment
npartitions=256,,,,,
,string,string,string,string,int64
,...,...,...,...,...
...,...,...,...,...,...
,...,...,...,...,...
,...,...,...,...,...


In [26]:
list_id = df1['id'].to_list()

In [22]:
df2 = df2.sample(frac=500000/len(df2), random_state=42).compute()

In [24]:
df2['domain'].value_counts()

domain
Health                       36508
News                         35024
Sensitive_Subjects           34667
Arts_and_Entertainment       34309
People_and_Society           30646
Business_and_Industrial      29418
Sports                       28998
Food_and_Drink               27101
Travel_and_Transportation    23711
Beauty_and_Fitness           23483
Home_and_Garden              22916
Jobs_and_Education           22307
Computers_and_Electronics    18619
Law_and_Government           17729
Real_Estate                  16283
Internet_and_Telecom         15304
Autos_and_Vehicles           15290
Finance                      14901
Books_and_Literature         13807
Games                        11951
Shopping                      8576
Pets_and_Animals              4966
Science                       4013
Hobbies_and_Leisure           3718
Adult                         3660
Online_Communities            2098
Name: count, dtype: int64[pyarrow]

In [32]:
list_ele = []
from tqdm import tqdm

for i in tqdm(range(len(df2))):
    ele = df2.iloc[i].to_dict()
    if ele['id'] in list_id:
        continue
    list_ele.append(ele)

100%|█████████████████████████████████████████████████████████████████████████| 500003/500003 [06:28<00:00, 1286.44it/s]


In [33]:
len(list_ele)

499627

In [34]:
df_new = pd.DataFrame(list_ele)

In [36]:
df_new = df_new.sample(frac=50000/len(df_new), random_state=42)

In [37]:
df_new

,filename,text,id,domain,sentiment
424138,62.parquet,"Posted on 26 Tháng Tư, 2018 by ngoc\nĐiều làm ...",VI_open-0098215668,Jobs_and_Education,0
434423,68.parquet,Cách bảo quản đồ dùng sơ sinh an toàn và hiệu ...,VI_open-0100838661,People_and_Society,1
340628,252.parquet,"Gay Tim Gay Ha Noi - phim Gay Tim Gay Ha Noi,x...",VI_open-0078114911,Adult,1
34161,112.parquet,"Khởi tố, bắt tạm giam đối tượng nhiều lần hủy ...",VI_open-0007332958,Law_and_Government,1
145097,161.parquet,Nhà lãnh đạo Triều Tiên Kim Jong-un thăm các đ...,VI_open-0031970234,News,1
...,...,...,...,...,...
322140,243.parquet,- Do tình hình giao thông Tp Hồ Chí Minh hiện ...,VI_open-0073410900,Travel_and_Transportation,0
13455,103.parquet,"Cho rằng tăng 8.000 đồng/lít quá cao, VINPA ""c...",VI_open-0003082469,Autos_and_Vehicles,1
64803,126.parquet,"Jean-Paul Sartre, người đầu tiên ""dám"" chê giả...",VI_open-0014269667,Arts_and_Entertainment,1
159183,169.parquet,"Chương trình ""Năm mới! Thêm bạn mới!"" | 12-201...",VI_open-0035805061,Finance,1


In [41]:
df_test = pd.concat([df_new, df1], axis=0)

In [44]:
df_test.drop(['quality_score'], axis=1, inplace=True)

In [45]:
df_test

,filename,text,id,domain,sentiment
424138,62.parquet,"Posted on 26 Tháng Tư, 2018 by ngoc\nĐiều làm ...",VI_open-0098215668,Jobs_and_Education,0
434423,68.parquet,Cách bảo quản đồ dùng sơ sinh an toàn và hiệu ...,VI_open-0100838661,People_and_Society,1
340628,252.parquet,"Gay Tim Gay Ha Noi - phim Gay Tim Gay Ha Noi,x...",VI_open-0078114911,Adult,1
34161,112.parquet,"Khởi tố, bắt tạm giam đối tượng nhiều lần hủy ...",VI_open-0007332958,Law_and_Government,1
145097,161.parquet,Nhà lãnh đạo Triều Tiên Kim Jong-un thăm các đ...,VI_open-0031970234,News,1
...,...,...,...,...,...
38576,99.parquet,"Tiền tệ | Thứ ba, 5/11/2019 | 14:34 GMT+7\nGiá...",VI_open-0116462023,Finance,1
36701,99.parquet,Viện kiểm sát nhân dân huyện Cẩm Mỹ thực hiện ...,VI_open-0116443985,Sensitive_Subjects,1
29212,99.parquet,Tp.Thanh Hóa - Tổ chức sự kiện chuyên nghiệp t...,VI_open-0116373951,Business_and_Industrial,1
31209,99.parquet,Video cách gửi tiền ngân hàng VN88 - Hướng dẫn...,VI_open-0116393187,Finance,0


In [47]:
df_test.to_csv('100k_heuristic_filtering.csv',index=False)

In [49]:
list_new_id = df_test['id'].to_list()

In [54]:
len(list_new_id)

99999

In [50]:
df3 = dd.read_parquet('/workspace/nemo_datasets/result')

In [52]:
df3 = df3.sample(frac=500000/len(df3), random_state=42).compute()

In [55]:
list_ele = []
from tqdm import tqdm

for i in tqdm(range(len(df3))):
    ele = df3.iloc[i].to_dict()
    if ele['id'] in list_new_id:
        continue
    list_ele.append(ele)

100%|██████████████████████████████████████████████████████████████████████████| 499968/499968 [34:33<00:00, 241.12it/s]


In [56]:
df4 = pd.DataFrame(list_ele)

In [60]:
df4

,filename,text,id,domain,sentiment
0,0.parquet,Khách sạn tại Phú Quốc TRANG CHỦ | GIỚI THIỆU ...,VI_open-0000367956,Travel_and_Transportation,1
1,0.parquet,Đẹp Online | Lạc vào bộ tộc… ở trần | Đẹp Onli...,VI_open-0000402059,Beauty_and_Fitness,1
2,0.parquet,Bảng giádịch vụ Spa Trang chủ / Tin tức / Lắng...,VI_open-0000442118,Beauty_and_Fitness,0
3,0.parquet,"Hôm nay 02/05/2022, đội ngũ chính thức niêm yế...",VI_open-0000137591,Finance,0
4,0.parquet,Thời khóa biểu và lịch thi Khóa 46 hệ liên kết...,VI_open-0000142830,Jobs_and_Education,1
...,...,...,...,...,...
499537,255.parquet,Thụy Điển tìm ra cách ngăn ngừa 100% tình trạn...,VI_open-0079544752,Health,0
499538,255.parquet,TPHCM: Thực hiện 21 chỉ tiêu trong năm 2020\nS...,VI_open-0079219242,Law_and_Government,1
499539,255.parquet,Xem Hồ sơ: ChaseJar - Cộng Đồng Mu Game Thủ\n0...,VI_open-0079455146,Games,0
499540,255.parquet,Bóng đá Việt Nam hôm nay. Hai năm kỳ tích của ...,VI_open-0079541345,Sports,0


In [62]:
df4 = df4.sample(frac=100000/len(df4), random_state=42)

In [66]:
df4

,filename,text,id,domain,sentiment
95712,49.parquet,"Thứ tư - 05/02/2014 22:28\nTừ khóa: nơi nơi, r...",VI_open-0091321030,Arts_and_Entertainment,0
193153,98.parquet,Nghề nuôi cá lồng bè trên sông Kinh Thày - ant...,VI_open-0116006798,Food_and_Drink,1
375229,192.parquet,Hat VS Buried Woman vui nhộn hài hước được yêu...,VI_open-0047614788,Arts_and_Entertainment,1
441402,226.parquet,Vòng phong thuỷ thạch anh đem lại tài lộc cho ...,VI_open-0065067034,Shopping,1
496314,254.parquet,Đăng bài Hôm qua lúc 05:25 PM bởi laji1110\nTư...,VI_open-0079088740,Home_and_Garden,1
...,...,...,...,...,...
431885,221.parquet,18/08/2018 10:23:54 (GMT+7)\n22/03/2018 03:16 ...,VI_open-0062549977,Business_and_Industrial,0
200974,102.parquet,Nhà cái V9BET - nhà cái cá độ bóng đá uy tín h...,VI_open-0002605821,Games,1
108861,55.parquet,Tìm thấy uranium dùng trong vũ khí ở Iran - Ti...,VI_open-0094535509,Sensitive_Subjects,1
171285,87.parquet,‎Bay Bay Kids on the App Store\nBay Bay Kids 4...,VI_open-0110314674,Jobs_and_Education,1


In [67]:
pd.merge(df_test, df4, on=['id'], how='inner')

,filename_x,text_x,id,domain_x,sentiment_x,filename_y,text_y,domain_y,sentiment_y


In [68]:
df_raw = pd.concat([df_test, df4], axis=0)

In [70]:
df_raw.to_csv('200k_raw.csv',index=False)

In [ ]:
df_test.to_csv('100k_heuristic_filtering.csv',index=False)

In [75]:
df_check = pd.read_csv('50k_classifier_filtering.csv')

In [76]:
df_check

,filename,text,id,domain,sentiment,quality_score
0,0.parquet,Một số nội dung sinh hoạt của cộng đồng người ...,VI_open-0000395764,People_and_Society,1,0.070625
1,0.parquet,Mũ bảo hiểm bóng đá mini là vật dụng lý tưởng ...,VI_open-0000093703,Sports,1,0.755936
2,0.parquet,Anh ngồi bấu tay vào thành giường nhìn ra ngoà...,VI_open-0000107371,Sensitive_Subjects,1,0.401747
3,0.parquet,Ai giỏi design muốn giành sách Nhật nào? [Lưu ...,VI_open-0000431777,Books_and_Literature,1,0.437828
4,0.parquet,Những đặc trưng cơ bản của quần thể phấn 1\nCậ...,VI_open-0000415391,Science,1,0.769130
...,...,...,...,...,...,...
49994,99.parquet,"Tiền tệ | Thứ ba, 5/11/2019 | 14:34 GMT+7\nGiá...",VI_open-0116462023,Finance,1,0.036170
49995,99.parquet,Viện kiểm sát nhân dân huyện Cẩm Mỹ thực hiện ...,VI_open-0116443985,Sensitive_Subjects,1,0.021462
49996,99.parquet,Tp.Thanh Hóa - Tổ chức sự kiện chuyên nghiệp t...,VI_open-0116373951,Business_and_Industrial,1,0.050194
49997,99.parquet,Video cách gửi tiền ngân hàng VN88 - Hướng dẫn...,VI_open-0116393187,Finance,0,0.002124


In [11]:
df_test = df[df['domain'] == 'Sensitive_Subjects'].compute()

In [7]:
df['sentiment'].value_counts().compute()

sentiment
1    11497524
0      690087
2       71843
Name: count, dtype: int64

In [8]:
df['domain'].value_counts().compute()

domain
Arts_and_Entertainment       941159
Health                       886657
People_and_Society           843624
Sports                       793235
News                         769510
Sensitive_Subjects           743582
Food_and_Drink               675447
Business_and_Industrial      624991
Travel_and_Transportation    581803
Jobs_and_Education           564654
Beauty_and_Fitness           555714
Home_and_Garden              529616
Computers_and_Electronics    516543
Books_and_Literature         396770
Internet_and_Telecom         379943
Law_and_Government           358097
Games                        345954
Autos_and_Vehicles           328610
Real_Estate                  319040
Finance                      304233
Pets_and_Animals             189001
Shopping                     187516
Science                      182682
Hobbies_and_Leisure           98387
Adult                         90323
Online_Communities            52363
Name: count, dtype: int64[pyarrow]

In [9]:
# Define the function to count words
def count_words(df):
    return df['text'].str.split().str.len()

# Apply the function to the DataFrame
df['word count'] = df.map_partitions(lambda df: count_words(df), meta=('word count', 'int64'))

# Compute the result to see the output
df.compute()

,filename,text,id,domain,sentiment,quality_score,word count
0,0.parquet,Internet Society hay ISOC là một tổ chức quốc ...,VI_open-0000000000,Internet_and_Telecom,1,0.965412,243
1,0.parquet,Tế bào là một đơn vị cấu trúc cơ bản có chức n...,VI_open-0000000001,Science,1,0.890338,4529
2,0.parquet,Quẻ Phong Thiên Tiểu Súcđồ hình |||:|| còn gọi...,VI_open-0000000002,Books_and_Literature,1,0.798719,184
3,0.parquet,Google LLC () là một công ty công nghệ đa quốc...,VI_open-0000000003,News,1,0.678852,9161
4,0.parquet,Nông thôn Việt Nam là một khái niệm chung dùng...,VI_open-0000000005,People_and_Society,1,0.606397,3442
...,...,...,...,...,...,...,...
45779,99.parquet,người thích đậu xanh thì có thể cho Răng không...,VI_open-0116529541,Health,1,0.246219,203
45780,99.parquet,Hộ tâm – Chương 6 | Thương thành Đang tung tăn...,VI_open-0116529549,Books_and_Literature,1,0.271915,2795
45781,99.parquet,Những lý do iPhone 8 cũ 128 Gb sẽ là chiếc điệ...,VI_open-0116529556,Internet_and_Telecom,1,0.273157,1453
45782,99.parquet,Percy Jackson & kẻ cắp tia chớp (Percy Jackson...,VI_open-0116529566,Arts_and_Entertainment,1,0.730473,282


In [6]:
import gc

gc.collect()

1405